# Tworzymy nazwane klastry

W lekcji tej użyjemy LLM-ów do pomocy przy znalezieniu odpowiednich nazw i opisów dla klastrów.

In [1]:
import json
import pandas as pd
from openai import OpenAI
from dotenv import dotenv_values
from pycaret.clustering import predict_model, load_model

In [2]:
env = dotenv_values(".env")

openai_client = OpenAI(api_key=env["OPENAI_API_KEY"])

ładujemy nasze dane

In [3]:
df = pd.read_csv('welcome_survey_simple_v2.csv', sep=';')
df.head()

,age,edu_level,fav_animals,fav_place,gender
0,<18,Podstawowe,Brak ulubionych,NaN,Kobieta
1,25-34,Średnie,Psy,Nad wodą,Mężczyzna
2,45-54,Wyższe,Psy,W lesie,Mężczyzna
3,35-44,Średnie,Koty,W górach,Mężczyzna
4,35-44,Wyższe,Psy,Nad wodą,Mężczyzna


ładujemy model wytrenowany w poprzedniej lekcji

In [4]:
kmeans_pipeline = load_model('welcome_survey_clustering_pipeline_v2')

Transformation Pipeline and Model Successfully Loaded


aplikujemy model do danych

In [5]:
df_with_clusters = predict_model(model=kmeans_pipeline, data=df)
df_with_clusters["Cluster"].value_counts()

Cluster 0     44
Cluster 3     32
Cluster 1     32
Cluster 6     32
Cluster 2     20
Cluster 10    14
Cluster 4     14
Cluster 9      9
Cluster 7      8
Cluster 8      8
Cluster 11     8
Cluster 5      8
Name: Cluster, dtype: int64

stworzymy teraz prompt, który prześlemy do LLM-a w celu znalezienia odpowiednich nazw i opisów dla klastrów

In [6]:
cluster_descriptions = {}
for cluster_id in df_with_clusters['Cluster'].unique():
    cluster_df = df_with_clusters[df_with_clusters['Cluster'] == cluster_id]
    summary = ""
    for column in df_with_clusters:
        if column == 'Cluster':
            continue

        value_counts = cluster_df[column].value_counts()
        value_counts_str = ', '.join([f"{idx}: {cnt}" for idx, cnt in value_counts.items()])
        summary += f"{column} - {value_counts_str}\n"

    cluster_descriptions[cluster_id] = summary

In [8]:
print(cluster_descriptions["Cluster 10"])

age - 18-24: 4, 25-34: 4, 45-54: 3, 35-44: 2, >=65: 1, 55-64: 0, <18: 0, unknown: 0
edu_level - Średnie: 14, Podstawowe: 0, Wyższe: 0
fav_animals - Psy: 14, Brak ulubionych: 0, Inne: 0, Koty: 0, Koty i Psy: 0
fav_place - Nad wodą: 8, W górach: 4, W lesie: 1, Inne: 0
gender - Mężczyzna: 14, Kobieta: 0



In [9]:
prompt = "Użyliśmy algorytmu klastrowania."
for cluster_id, description in cluster_descriptions.items():
    prompt += f"\n\nKlaster {cluster_id}:\n{description}"

prompt += """
Wygeneruj najlepsze nazwy dla każdego z klasterów oraz ich opisy

Użyj formatu JSON. Przykładowo:
{
    "Cluster 0": {
        "name": "Klaster 0",
        "description": "W tym klastrze znajdują się osoby, które..."
    },
    "Cluster 1": {
        "name": "Klaster 1",
        "description": "W tym klastrze znajdują się osoby, które..."
    }
}
"""
print(prompt)

Użyliśmy algorytmu klastrowania.

Klaster Cluster 7:
age - 35-44: 6, 45-54: 1, <18: 1, 18-24: 0, 25-34: 0, 55-64: 0, >=65: 0, unknown: 0
edu_level - Średnie: 7, Podstawowe: 1, Wyższe: 0
fav_animals - Psy: 3, Brak ulubionych: 2, Koty: 2, Inne: 1, Koty i Psy: 0
fav_place - Nad wodą: 4, W lesie: 1, Inne: 0, W górach: 0
gender - Kobieta: 6, Mężczyzna: 2


Klaster Cluster 10:
age - 18-24: 4, 25-34: 4, 45-54: 3, 35-44: 2, >=65: 1, 55-64: 0, <18: 0, unknown: 0
edu_level - Średnie: 14, Podstawowe: 0, Wyższe: 0
fav_animals - Psy: 14, Brak ulubionych: 0, Inne: 0, Koty: 0, Koty i Psy: 0
fav_place - Nad wodą: 8, W górach: 4, W lesie: 1, Inne: 0
gender - Mężczyzna: 14, Kobieta: 0


Klaster Cluster 3:
age - 45-54: 19, 35-44: 8, 55-64: 4, >=65: 1, 18-24: 0, 25-34: 0, <18: 0, unknown: 0
edu_level - Wyższe: 32, Podstawowe: 0, Średnie: 0
fav_animals - Psy: 19, Brak ulubionych: 5, Koty: 5, Inne: 3, Koty i Psy: 0
fav_place - W lesie: 31, Inne: 1, Nad wodą: 0, W górach: 0
gender - Mężczyzna: 25, Kobieta: 7

In [10]:
response = openai_client.chat.completions.create(
    model="gpt-4o",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }
    ],
)

In [11]:
result = response.choices[0].message.content.replace("```json", "").replace("```", "").strip()
cluster_names_and_descriptions = json.loads(result)

In [12]:
with open("welcome_survey_cluster_names_and_descriptions_v2.json", "w") as f:
    f.write(json.dumps(cluster_names_and_descriptions))

In [13]:
with open("welcome_survey_cluster_names_and_descriptions_v2.json", "r") as f:
    print(json.loads(f.read()))

{'Cluster 0': {'name': 'Wodni Mężczyźni z Wyższym Wykształceniem', 'description': 'W tym klastrze znajdują się głównie mężczyźni w wieku 35-44 lat z wyższym wykształceniem, którzy preferują spędzanie czasu nad wodą i lubią psy.'}, 'Cluster 1': {'name': 'Górscy Profesjonaliści', 'description': 'W tym klastrze znajdują się osoby z wyższym wykształceniem, głównie mężczyźni w wieku 45-54 lat, którzy preferują góry i mają szczególne upodobanie do psów.'}, 'Cluster 2': {'name': 'Koci Miłośnicy Gór', 'description': 'W tym klastrze znajdują się osoby w wieku 35-44 lat z wyższym wykształceniem, które preferują góry i mają skłonność do lubienia kotów.'}, 'Cluster 3': {'name': 'Leśni Wykształceni', 'description': 'W tym klastrze znajdują się głównie mężczyźni w wieku 45-54 lat z wyższym wykształceniem, którzy preferują spędzanie czasu w lesie i mają różnorodne preferencje dotyczące zwierząt.'}, 'Cluster 4': {'name': 'Górscy Miłośnicy Zwierząt', 'description': 'W tym klastrze znajdują się głównie 